# Week 1: Probing Early Circuit Emergence

This notebook analyzes the intermediate activations of a small MLP trained on a synthetic task. The goal is to compare the representational dynamics of models trained with two different optimizers: AdamW and SGD.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

## 1. Load Activation Data

First, we need to load the saved activation tensors from the training runs. We'll organize them by optimizer and training step.

In [ ]:
def load_activations(base_path="../analysis/activations"): 
    """Loads all activation .npz files from the specified base path."""
    activation_data = {}
    optimizers = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    
    for opt in optimizers:
        opt_path = os.path.join(base_path, opt)
        activation_data[opt] = {}
        files = sorted([f for f in os.listdir(opt_path) if f.endswith('.npz')], key=lambda x: int(x.split('_')[1].split('.')[0]))
        
        for f in files:
            step = int(f.split('_')[1].split('.')[0])
            data = np.load(os.path.join(opt_path, f))
            activation_data[opt][step] = {key: data[key] for key in data}
            
    return activation_data

activation_data = load_activations()
print(f"Loaded optimizers: {list(activation_data.keys())}")
if 'adamw' in activation_data:
    print(f"AdamW steps: {list(activation_data['adamw'].keys())}")

## 2. Analyze Representational Alignment (Self-Similarity)

How do the representations within a single model evolve over training? We can measure this by computing the cosine similarity of a layer's activations at step `i` with its activations at step `j`. A high similarity suggests the geometry of the representation is stable.

In [ ]:
def analyze_self_similarity(data, optimizer, layer_name='act1'):
    """Computes the self-similarity matrix for a given layer's activations."""
    steps = sorted(data[optimizer].keys())
    num_steps = len(steps)
    similarity_matrix = np.zeros((num_steps, num_steps))

    for i in range(num_steps):
        for j in range(num_steps):
            step_i = steps[i]
            step_j = steps[j]
            
            # Flatten the activations to (batch_size * seq_len, hidden_size)
            act_i = data[optimizer][step_i][layer_name].reshape(-1, data[optimizer][step_i][layer_name].shape[-1])
            act_j = data[optimizer][step_j][layer_name].reshape(-1, data[optimizer][step_j][layer_name].shape[-1])
            
            # Use the mean of pairwise similarities for a stable measure
            similarity = cosine_similarity(act_i, act_j).mean()
            similarity_matrix[i, j] = similarity
            
    return similarity_matrix, steps

if 'adamw' in activation_data:
    adamw_sim_act1, steps = analyze_self_similarity(activation_data, 'adamw', layer_name='act1')

    plt.figure(figsize=(8, 6))
    sns.heatmap(adamw_sim_act1, xticklabels=steps, yticklabels=steps, annot=True, fmt=".2f", cmap="viridis")
    plt.title("AdamW: Self-Similarity of Layer 1 (act1) Activations")
    plt.xlabel("Training Step")
    plt.ylabel("Training Step")
    plt.show()

## 3. Cross-Optimizer Alignment

This is the core question: do AdamW and SGD learn similar functions? We can probe this by comparing the activations from the two models at the same training step.

In [ ]:
def analyze_cross_optimizer_similarity(data, layer_name='act1'):
    """Compares activations between AdamW and SGD at each corresponding step."""
    if 'adamw' not in data or 'sgd' not in data:
        print("Data for both optimizers is required for this analysis.")
        return None, None
    
    adam_steps = sorted(data['adamw'].keys())
    sgd_steps = sorted(data['sgd'].keys())
    common_steps = sorted(list(set(adam_steps) & set(sgd_steps)))
    
    similarities = []
    for step in common_steps:
        act_adam = data['adamw'][step][layer_name].reshape(-1, data['adamw'][step][layer_name].shape[-1])
        act_sgd = data['sgd'][step][layer_name].reshape(-1, data['sgd'][step][layer_name].shape[-1])
        
        similarity = cosine_similarity(act_adam, act_sgd).mean()
        similarities.append(similarity)
        
    return similarities, common_steps

# Before running this, we need to generate data for the SGD optimizer.
# Example plotting code is below for when SGD data is available.
similarities, common_steps = analyze_cross_optimizer_similarity(activation_data)

if similarities:
    plt.figure(figsize=(10, 5))
    plt.plot(common_steps, similarities, marker='o', linestyle='-')
    plt.title("Cross-Optimizer (AdamW vs. SGD) Representational Alignment")
    plt.xlabel("Training Step")
    plt.ylabel("Mean Cosine Similarity")
    plt.ylim(0, 1)
    plt.show()